# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.4 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:

TASK_ID = "task044"
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path("/mnt/data/task044.json")
KAGGLE_TASK_JSON = Path(COMPETITION) / f"{TASK_ID}.json"
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
OUT_DIR = Path.cwd() / f"{TASK_ID}_static_model_onnx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"
with TASK_JSON.open('r') as f:
    task=json.load(f)
print(TASK_ID, len(task.get('train', [])), len(task.get('test', [])), len(task.get('arc-gen', [])))


task044 3 1 262


In [6]:

def grid_to_tensor(grid, h=H, w=W, ch=CH, full_background=False):
    x=np.zeros((1,ch,h,w),dtype=np.float32)
    if full_background:
        x[0,0,:,:]=1.0
    for r,row in enumerate(grid):
        for c,v in enumerate(row):
            if full_background:
                x[0,:,r,c]=0.0
            x[0,int(v),r,c]=1.0
    return x

def tensor_to_grid(y, h, w):
    return y[0,:,:h,:w].argmax(axis=0).astype(int).tolist()

def expected_tensor(ex):
    return grid_to_tensor(ex['output'], full_background=False)


In [7]:

class Task044FullCanvasMomentHoleFill(nn.Module):
    """Full-canvas component-aware hole fill without visible-template memorization.

    Structural rule:
    - Color 5 forms closed frames. Zero cells with 5 on both row/column sides are candidate frame holes.
    - The two hole components are isolated by static unrolled 4-neighbor propagation.
    - Each non-0/non-5 color outside the frame is represented by translation-invariant shape moments.
    - A hole is filled with the color whose component has the same area, bbox, and normalized moments.
    - Matched source components are removed; distractors and markers remain unchanged.

    This operates over the full 30x30 padded tensor. The active canvas is detected from input channels;
    output outside the active canvas is forced to all-zero.
    """
    def __init__(self, iters=30):
        super().__init__()
        self.iters=iters
        self.register_buffer('k4', torch.tensor([[[[0.,1.,0.],[1.,1.,1.],[0.,1.,0.]]]], dtype=torch.float32))
        rr=torch.arange(30,dtype=torch.float32).view(1,1,30,1).expand(1,1,30,30)
        cc=torch.arange(30,dtype=torch.float32).view(1,1,1,30).expand(1,1,30,30)
        self.register_buffer('rr',rr)
        self.register_buffer('cc',cc)
        self.register_buffer('big', torch.tensor(1000.0, dtype=torch.float32))

    def flood(self, seed, allowed):
        m=seed*allowed
        for _ in range(self.iters):
            m=(F.conv2d(m,self.k4,padding=1)>0.5).float()*allowed
        return m

    def first_component(self, mask):
        flat=mask.reshape(1,1,900)
        cs=torch.cumsum(flat,dim=2)
        seed=(flat>0.5).float()*(cs<1.5).float()
        return self.flood(seed.reshape(1,1,30,30), mask)

    def shape_features(self, m):
        n=m.sum(dim=(2,3),keepdim=True)
        exists=(n>0.5).float()
        rmin=(m*self.rr+(1-m)*self.big).amin(dim=(2,3),keepdim=True)
        cmin=(m*self.cc+(1-m)*self.big).amin(dim=(2,3),keepdim=True)
        rmax=(m*self.rr).amax(dim=(2,3),keepdim=True)
        cmax=(m*self.cc).amax(dim=(2,3),keepdim=True)
        dr=(self.rr-rmin)*m
        dc=(self.cc-cmin)*m
        return [
            n,
            (rmax-rmin+1)*exists,
            (cmax-cmin+1)*exists,
            dr.sum(dim=(2,3),keepdim=True),
            dc.sum(dim=(2,3),keepdim=True),
            (dr*dr).sum(dim=(2,3),keepdim=True),
            (dc*dc).sum(dim=(2,3),keepdim=True),
            (dr*dc).sum(dim=(2,3),keepdim=True),
            (dr*dr*dc).sum(dim=(2,3),keepdim=True),
            (dr*dc*dc).sum(dim=(2,3),keepdim=True),
            (dr*dr*dr).sum(dim=(2,3),keepdim=True),
            (dc*dc*dc).sum(dim=(2,3),keepdim=True),
        ]

    def same_shape(self, a, b):
        s=torch.ones_like(a[0])
        for x,y in zip(a,b):
            s=s*(torch.abs(x-y)<0.5).float()
        return s

    def forward(self, x):
        active=(x.sum(dim=1,keepdim=True)>0.5).float()
        zero=x[:,0:1]*active
        frame=x[:,5:6]*active

        # Candidate holes are zero cells boxed by color-5 in both axes.
        # This intentionally avoids treating padded 0 as background: candidate holes are inside active canvas only.
        left=torch.cumsum(frame,dim=3)-frame
        right=torch.flip(torch.cumsum(torch.flip(frame,[3]),dim=3),[3])-frame
        up=torch.cumsum(frame,dim=2)-frame
        down=torch.flip(torch.cumsum(torch.flip(frame,[2]),dim=2),[2])-frame
        holes=zero*(left>0.5).float()*(right>0.5).float()*(up>0.5).float()*(down>0.5).float()

        h1=self.first_component(holes)
        h2=self.first_component(holes*(1-h1))
        hole_masks=[h1,h2]
        hole_features=[self.shape_features(h1), self.shape_features(h2)]

        fill=[torch.zeros_like(zero) for _ in range(10)]
        used=[torch.zeros_like(zero) for _ in range(10)]
        for col in [1,2,3,4,6,7,8,9]:
            src=x[:,col:col+1]*active
            src_features=self.shape_features(src)
            matched=torch.zeros_like(src_features[0])
            for hm,hf in zip(hole_masks,hole_features):
                match=self.same_shape(hf,src_features)*(hf[0]>0.5).float()*(src_features[0]>0.5).float()
                fill[col]=torch.clamp(fill[col]+hm*match,0,1)
                matched=torch.clamp(matched+match,0,1)
            used[col]=src*matched

        channels=[None]*10
        occ=torch.zeros_like(zero)
        for col in range(1,10):
            orig=x[:,col:col+1]*active
            if col==5:
                outc=orig
            else:
                outc=torch.clamp(orig*(1-used[col])+fill[col],0,1)
            channels[col]=outc
            occ=torch.clamp(occ+outc,0,1)
        channels[0]=active*(1-occ)
        return torch.cat(channels,dim=1)*active

model=Task044FullCanvasMomentHoleFill().eval()


In [8]:

dummy = torch.from_numpy(grid_to_tensor(task['test'][0]['input']))
torch.onnx.export(
    model, dummy, str(ONNX_PATH), input_names=['input'], output_names=['output'],
    opset_version=17, do_constant_folding=True, dynamic_axes=None, dynamo=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))
print('ONNX:', ONNX_PATH, 'bytes:', ONNX_PATH.stat().st_size)


/tmp/ipykernel_16/1118339530.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX: /kaggle/working/task044_static_model_onnx/task044.onnx bytes: 323513


In [9]:

def vi_shape(vi):
    return [int(d.dim_value) if d.dim_value else (d.dim_param or None) for d in vi.type.tensor_type.shape.dim]
onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
summary_static = {
    'input_shape': vi_shape(onnx_model.graph.input[0]),
    'output_shape': vi_shape(onnx_model.graph.output[0]),
    'onnx_size_bytes': ONNX_PATH.stat().st_size,
    'ops': dict(ops),
    'forbidden_ops': sorted(forbidden & set(ops)),
}
print(json.dumps(summary_static, indent=2)[:4000])
assert summary_static['input_shape'] == [1,10,30,30]
assert summary_static['output_shape'] == [1,10,30,30]
assert summary_static['onnx_size_bytes'] < 1_400_000
assert not summary_static['forbidden_ops']


{
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 323513,
  "ops": {
    "Constant": 509,
    "ReduceSum": 101,
    "Greater": 77,
    "Cast": 271,
    "Slice": 14,
    "Mul": 477,
    "CumSum": 6,
    "Sub": 255,
    "Reshape": 4,
    "Less": 194,
    "Conv": 60,
    "Add": 89,
    "ReduceMin": 20,
    "ReduceMax": 20,
    "Abs": 192,
    "Clip": 49,
    "Concat": 1
  },
  "forbidden_ops": []
}


In [10]:

sess_options = ort.SessionOptions(); sess_options.intra_op_num_threads=1; sess_options.inter_op_num_threads=1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=['CPUExecutionProvider'])

def validate_examples(examples, full_background=False):
    ok=0; bad=[]; outside_zero_ok=0; active_exact_ok=0
    for i,ex in enumerate(examples):
        x=grid_to_tensor(ex['input'], full_background=full_background)
        y=sess.run(None, {'input':x})[0]
        pred=(y>0.5).astype(np.float32)
        exp=expected_tensor(ex)
        if np.array_equal(pred, exp):
            ok += 1
        else:
            bad.append(i)
        active=(x.sum(axis=1,keepdims=True)>0.5).astype(np.float32)
        outside_zero_ok += bool(np.all(pred*(1-active)==0))
        active_exact_ok += bool(np.array_equal(pred*active, exp*active))
    return {'ok':ok, 'total':len(examples), 'bad_first10':bad[:10], 'outside_zero_ok':outside_zero_ok, 'active_exact_ok':active_exact_ok}

rng=random.Random(0)
inds=list(range(len(task.get('arc-gen', []))))
rng.shuffle(inds)
hold=[task['arc-gen'][i] for i in inds[:math.ceil(0.6*len(inds))]] if inds else []
validation={
    'strict_zero_padding': {
        'train': validate_examples(task['train'], False),
        'test': validate_examples(task['test'], False),
        'arc_gen_60pct_holdout': validate_examples(hold, False) if hold else None,
        'arc_gen_full_diagnostic': validate_examples(task.get('arc-gen', []), False) if task.get('arc-gen') else None,
    },
    'full_background_padding_diagnostic': {
        'train': validate_examples(task['train'], True),
        'test': validate_examples(task['test'], True),
    }
}
summary={'task_id':TASK_ID, 'model_class':model.__class__.__name__, **summary_static, 'validation':validation}
json.dump(summary, open(SUMMARY_PATH,'w'), indent=2)
print(json.dumps(summary, indent=2)[:5000])
assert validation['strict_zero_padding']['train']['ok'] == validation['strict_zero_padding']['train']['total']
assert validation['strict_zero_padding']['test']['ok'] == validation['strict_zero_padding']['test']['total']
if hold:
    assert validation['strict_zero_padding']['arc_gen_60pct_holdout']['ok'] == validation['strict_zero_padding']['arc_gen_60pct_holdout']['total']


{
  "task_id": "task044",
  "model_class": "Task044FullCanvasMomentHoleFill",
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 323513,
  "ops": {
    "Constant": 509,
    "ReduceSum": 101,
    "Greater": 77,
    "Cast": 271,
    "Slice": 14,
    "Mul": 477,
    "CumSum": 6,
    "Sub": 255,
    "Reshape": 4,
    "Less": 194,
    "Conv": 60,
    "Add": 89,
    "ReduceMin": 20,
    "ReduceMax": 20,
    "Abs": 192,
    "Clip": 49,
    "Concat": 1
  },
  "forbidden_ops": [],
  "validation": {
    "strict_zero_padding": {
      "train": {
        "ok": 3,
        "total": 3,
        "bad_first10": [],
        "outside_zero_ok": 3,
        "active_exact_ok": 3
      },
      "test": {
        "ok": 1,
        "total": 1,
        "bad_first10": [],
        "outside_zero_ok": 1,
        "active_exact_ok": 1
      },
      "arc_gen_60pct_holdout": {
        "ok": 158,
        "total": 158,
        "bad_first10": [],

In [11]:

with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
print('Wrote:', SUBMISSION_PATH)
print('Zip contents:', zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f'{TASK_ID}.onnx']


Wrote: /kaggle/working/submission.zip
Zip contents: ['task044.onnx']
